# 🏦 Banking Credit Risk & Fraud Detection Analytics

## Notebook 08 — SQL & Business Analytics

### Objective

The objective of this notebook is to perform business-oriented analysis using SQL concepts on the credit-risk and fraud datasets.

The analysis will translate raw transaction and loan data into business insights that can support:

- Credit risk management
- Loan portfolio monitoring
- Customer segmentation
- Default-risk analysis
- Fraud monitoring
- Operational decision-making

### Key Business Questions

The analysis will investigate questions such as:

1. Which customer segments have the highest loan-default rates?
2. Which loan purposes have elevated default risk?
3. How does loan grade relate to default risk?
4. How does home ownership relate to loan outcomes?
5. Which customer characteristics are associated with higher loan risk?
6. What patterns can be observed in fraudulent transactions?
7. How can analytics and machine-learning predictions support banking decisions?

### Analytical Approach

The workflow will combine:

- SQL-style data aggregation
- Descriptive statistics
- Business KPIs
- Customer segmentation
- Risk analysis
- Fraud analysis
- Machine-learning findings

The objective is to convert technical analysis into actionable business recommendations.

## 1. Load Credit Risk Dataset

The Credit Risk dataset contains applicant-level information including:

- Age
- Income
- Employment length
- Home ownership
- Loan intent
- Loan grade
- Loan amount
- Interest rate
- Loan-to-income ratio
- Credit history length
- Previous credit default indicator
- Loan status

The `loan_status` variable represents the loan outcome:

- `0` = Non-default
- `1` = Default

This dataset will be used to answer business questions related to lending risk and portfolio quality.

In [23]:
import pandas as pd
import numpy as np

In [5]:
from pathlib import Path

project_path = Path(
    r"D:\Banking-Credit-Risk-Fraud-Analytics"
)

csv_files = list(
    project_path.rglob("*.csv")
)

print("CSV files found:\n")

for file in csv_files:
    print(file)

CSV files found:

D:\Banking-Credit-Risk-Fraud-Analytics\data\credit_risk\credit_risk_dataset.csv
D:\Banking-Credit-Risk-Fraud-Analytics\data\fraud\creditcard.csv
D:\Banking-Credit-Risk-Fraud-Analytics\data\Processed\credit_risk_cleaned.csv
D:\Banking-Credit-Risk-Fraud-Analytics\data\Processed\fraud_cleaned.csv


In [6]:
credit_df = pd.read_csv(
    r"D:\Banking-Credit-Risk-Fraud-Analytics\data\credit_risk\credit_risk_dataset.csv"
)

print("Credit dataset shape:", credit_df.shape)

print("\nColumns:")
print(credit_df.columns.tolist())

Credit dataset shape: (32581, 12)

Columns:
['person_age', 'person_income', 'person_home_ownership', 'person_emp_length', 'loan_intent', 'loan_grade', 'loan_amnt', 'loan_int_rate', 'loan_status', 'loan_percent_income', 'cb_person_default_on_file', 'cb_person_cred_hist_length']


In [7]:
credit_df.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


In [8]:
credit_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32581 entries, 0 to 32580
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   person_age                  32581 non-null  int64  
 1   person_income               32581 non-null  int64  
 2   person_home_ownership       32581 non-null  object 
 3   person_emp_length           31686 non-null  float64
 4   loan_intent                 32581 non-null  object 
 5   loan_grade                  32581 non-null  object 
 6   loan_amnt                   32581 non-null  int64  
 7   loan_int_rate               29465 non-null  float64
 8   loan_status                 32581 non-null  int64  
 9   loan_percent_income         32581 non-null  float64
 10  cb_person_default_on_file   32581 non-null  object 
 11  cb_person_cred_hist_length  32581 non-null  int64  
dtypes: float64(3), int64(5), object(4)
memory usage: 3.0+ MB


In [9]:
credit_df["loan_status"].value_counts()

loan_status
0    25473
1     7108
Name: count, dtype: int64

## 1.1 Data Quality Check

Before performing business analysis, the dataset is checked for:

- Missing values
- Duplicate records
- Data types
- Invalid or unexpected values

This ensures that the business metrics calculated in subsequent sections are based on reliable data.

In [10]:
print("Dataset shape:", credit_df.shape)

print("\nMissing values:")
print(credit_df.isnull().sum())

print("\nDuplicate rows:")
print(credit_df.duplicated().sum())

print("\nData types:")
print(credit_df.dtypes)

Dataset shape: (32581, 12)

Missing values:
person_age                       0
person_income                    0
person_home_ownership            0
person_emp_length              895
loan_intent                      0
loan_grade                       0
loan_amnt                        0
loan_int_rate                 3116
loan_status                      0
loan_percent_income              0
cb_person_default_on_file        0
cb_person_cred_hist_length       0
dtype: int64

Duplicate rows:
165

Data types:
person_age                      int64
person_income                   int64
person_home_ownership          object
person_emp_length             float64
loan_intent                    object
loan_grade                     object
loan_amnt                       int64
loan_int_rate                 float64
loan_status                     int64
loan_percent_income           float64
cb_person_default_on_file      object
cb_person_cred_hist_length      int64
dtype: object


## 2. Loan Portfolio Overview

The first business analysis establishes the overall size and risk profile of the lending portfolio.

The key portfolio KPIs are:

- Total number of loans
- Total loan exposure
- Average loan amount
- Average interest rate
- Overall default rate

These metrics provide a high-level view of portfolio scale and credit performance.

In [11]:
portfolio_kpis = pd.DataFrame({
    "KPI": [
        "Total Loans",
        "Total Loan Exposure",
        "Average Loan Amount",
        "Average Interest Rate",
        "Overall Default Rate"
    ],
    
    "Value": [
        len(credit_df),
        credit_df["loan_amnt"].sum(),
        credit_df["loan_amnt"].mean(),
        credit_df["loan_int_rate"].mean(),
        credit_df["loan_status"].mean()
    ]
})

portfolio_kpis

,KPI,Value
0,Total Loans,3.258100e+04
1,Total Loan Exposure,3.124313e+08
2,Average Loan Amount,9.589371e+03
3,Average Interest Rate,1.101169e+01
4,Overall Default Rate,2.181640e-01


In [12]:
print(
    f"Overall Default Rate: "
    f"{credit_df['loan_status'].mean():.2%}"
)

Overall Default Rate: 21.82%


## 3. Default Risk by Loan Grade

Loan grade is used to segment borrowers according to their observed credit-risk profile.

The analysis calculates:

- Number of loans
- Number of defaults
- Default rate
- Average loan amount
- Average interest rate

The objective is to identify whether default risk is concentrated in particular loan grades.

In [13]:
grade_analysis = (
    credit_df
    .groupby("loan_grade")
    .agg(
        total_loans=("loan_status", "count"),
        defaults=("loan_status", "sum"),
        default_rate=("loan_status", "mean"),
        avg_loan_amount=("loan_amnt", "mean"),
        avg_interest_rate=("loan_int_rate", "mean")
    )
    .reset_index()
)

grade_analysis

,loan_grade,total_loans,defaults,default_rate,avg_loan_amount,avg_interest_rate
0,A,10777,1073,0.099564,8539.273453,7.327651
1,B,10451,1701,0.162760,9995.483686,10.995555
2,C,6458,1339,0.207340,9213.862651,13.463542
3,D,3626,2141,0.590458,10849.241589,15.361448
4,E,964,621,0.644191,12915.845436,17.009455
5,F,241,170,0.705394,14717.323651,18.609159
6,G,64,63,0.984375,17195.703125,20.251525


In [14]:
grade_analysis.sort_values(
    "default_rate",
    ascending=False
)

,loan_grade,total_loans,defaults,default_rate,avg_loan_amount,avg_interest_rate
6,G,64,63,0.984375,17195.703125,20.251525
5,F,241,170,0.705394,14717.323651,18.609159
4,E,964,621,0.644191,12915.845436,17.009455
3,D,3626,2141,0.590458,10849.241589,15.361448
2,C,6458,1339,0.207340,9213.862651,13.463542
1,B,10451,1701,0.162760,9995.483686,10.995555
0,A,10777,1073,0.099564,8539.273453,7.327651


## 4. Default Risk by Loan Intent

Loan intent represents the purpose for which the borrower is requesting the loan.

Default rates will be compared across loan purposes to identify segments with relatively higher observed credit risk.

This analysis can help lenders understand whether certain loan purposes require additional monitoring or underwriting attention.

In [15]:
intent_analysis = (
    credit_df
    .groupby("loan_intent")
    .agg(
        total_loans=("loan_status", "count"),
        defaults=("loan_status", "sum"),
        default_rate=("loan_status", "mean"),
        avg_loan_amount=("loan_amnt", "mean")
    )
    .reset_index()
)

intent_analysis.sort_values(
    "default_rate",
    ascending=False
)

,loan_intent,total_loans,defaults,default_rate,avg_loan_amount
0,DEBTCONSOLIDATION,5212,1490,0.285879,9594.886800
3,MEDICAL,6071,1621,0.267007,9259.582441
2,HOMEIMPROVEMENT,3605,941,0.261026,10360.520111
4,PERSONAL,5521,1098,0.198877,9573.772867
1,EDUCATION,6453,1111,0.172168,9482.678599
5,VENTURE,5719,847,0.148103,9583.777758


## 5. Default Risk by Home Ownership

Home ownership provides another customer-level segmentation variable.

Default rates will be compared across different ownership categories to identify differences in observed loan performance.

The analysis also compares average income and average loan amount across these segments.

In [16]:
ownership_analysis = (
    credit_df
    .groupby("person_home_ownership")
    .agg(
        total_loans=("loan_status", "count"),
        defaults=("loan_status", "sum"),
        default_rate=("loan_status", "mean"),
        avg_income=("person_income", "mean"),
        avg_loan_amount=("loan_amnt", "mean")
    )
    .reset_index()
)

ownership_analysis.sort_values(
    "default_rate",
    ascending=False
)

,person_home_ownership,total_loans,defaults,default_rate,avg_income,avg_loan_amount
3,RENT,16446,5192,0.315700,54997.747963,8862.331266
1,OTHER,107,33,0.308411,76387.803738,11074.532710
0,MORTGAGE,13444,1690,0.125707,81127.121690,10574.460726
2,OWN,2584,193,0.074690,57834.812693,9029.943885


In [17]:
business_risk_summary = pd.DataFrame({
    "Analysis": [
        "Highest Risk Loan Grade",
        "Highest Risk Loan Intent",
        "Highest Risk Home Ownership"
    ],
    
    "Segment": [
        grade_analysis.loc[
            grade_analysis["default_rate"].idxmax(),
            "loan_grade"
        ],
        
        intent_analysis.loc[
            intent_analysis["default_rate"].idxmax(),
            "loan_intent"
        ],
        
        ownership_analysis.loc[
            ownership_analysis["default_rate"].idxmax(),
            "person_home_ownership"
        ]
    ],
    
    "Default Rate": [
        grade_analysis["default_rate"].max(),
        intent_analysis["default_rate"].max(),
        ownership_analysis["default_rate"].max()
    ]
})

business_risk_summary

,Analysis,Segment,Default Rate
0,Highest Risk Loan Grade,G,0.984375
1,Highest Risk Loan Intent,DEBTCONSOLIDATION,0.285879
2,Highest Risk Home Ownership,RENT,0.315700


## 5.1 Initial Business Findings

The portfolio analysis provides several important risk-segmentation insights.

Loan grade shows substantial variation in observed default rates, indicating that credit quality is an important portfolio-risk differentiator.

Loan intent also shows differences in default rates across borrowing purposes. These differences can help identify segments requiring additional risk monitoring.

Home ownership categories also demonstrate different observed default rates, suggesting that customer characteristics can contribute to portfolio segmentation.

These relationships are descriptive and should not be interpreted as causal relationships.

The findings can nevertheless support:
`
- Risk-based customer segmentation
- Portfolio monitoring
- Underwriting review
- Risk-based pricing
- Targeted credit-risk management

The machine-learning models developed earlier can complement these descriptive analyses by combining multiple variables simultaneously.

In [19]:
credit_analysis_df = credit_df.copy()

print(
    "Analysis dataframe created:",
    credit_analysis_df.shape
)

Analysis dataframe created: (32581, 12)


In [20]:
credit_analysis_df

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4
...,...,...,...,...,...,...,...,...,...,...,...,...
32576,57,53000,MORTGAGE,1.0,PERSONAL,C,5800,13.16,0,0.11,N,30
32577,54,120000,MORTGAGE,4.0,PERSONAL,A,17625,7.49,0,0.15,N,19
32578,65,76000,RENT,3.0,HOMEIMPROVEMENT,B,35000,10.99,1,0.46,N,28
32579,56,150000,MORTGAGE,5.0,PERSONAL,B,15000,11.48,0,0.10,N,26


## 6. Default Risk by Income Segment

Borrower income is an important lending-risk variable because it provides an indication of repayment capacity.

Instead of analyzing income only as a continuous variable, borrowers will be grouped into four income segments.

The default rate, average income, and average loan amount will then be compared across these segments.

The objective is to determine whether observed default risk differs across borrower income groups.

In [21]:
credit_analysis_df["income_segment"] = pd.qcut(
    credit_analysis_df["person_income"],
    q=4,
    labels=[
        "Low Income",
        "Lower-Middle Income",
        "Upper-Middle Income",
        "High Income"
    ]
)

income_analysis = (
    credit_analysis_df
    .groupby(
        "income_segment",
        observed=True
    )
    .agg(
        total_loans=("loan_status", "count"),
        defaults=("loan_status", "sum"),
        default_rate=("loan_status", "mean"),
        avg_income=("person_income", "mean"),
        avg_loan_amount=("loan_amnt", "mean")
    )
    .reset_index()
)

income_analysis

,income_segment,total_loans,defaults,default_rate,avg_income,avg_loan_amount
0,Low Income,8152,3236,0.396958,28615.324092,6308.234176
1,Lower-Middle Income,8231,1755,0.213218,47041.932815,8499.614263
2,Upper-Middle Income,8055,1375,0.170701,66282.422595,10453.438858
3,High Income,8143,742,0.091121,122609.044947,13120.938229


## 7. Default Risk by Loan-to-Income Ratio

The loan-to-income ratio represents the relative size of the requested loan compared with the borrower's income.

A higher ratio may indicate greater repayment pressure because the loan represents a larger proportion of the borrower's income.

Borrowers will therefore be grouped into loan-to-income bands and their observed default rates will be compared.

In [24]:
credit_analysis_df["loan_income_segment"] = pd.cut(
    credit_analysis_df["loan_percent_income"],
    bins=[
        -np.inf,
        0.10,
        0.20,
        0.30,
        0.40,
        np.inf
    ],
    labels=[
        "<=10%",
        "10%-20%",
        "20%-30%",
        "30%-40%",
        ">40%"
    ]
)

loan_income_analysis = (
    credit_analysis_df
    .groupby(
        "loan_income_segment",
        observed=True
    )
    .agg(
        total_loans=("loan_status", "count"),
        defaults=("loan_status", "sum"),
        default_rate=("loan_status", "mean"),
        avg_loan_amount=("loan_amnt", "mean")
    )
    .reset_index()
)

loan_income_analysis

,loan_income_segment,total_loans,defaults,default_rate,avg_loan_amount
0,<=10%,10484,1229,0.117226,5253.381343
1,10%-20%,12066,1823,0.151086,9646.929388
2,20%-30%,6197,1360,0.219461,12978.864773
3,30%-40%,2714,1865,0.687178,15136.560427
4,>40%,1120,831,0.741964,17361.026786


## 6. Default Risk by Income Segment

Borrower income is an important lending-risk variable because it provides an indication of repayment capacity.

Instead of analyzing income only as a continuous variable, borrowers will be grouped into four income segments.

The default rate, average income, and average loan amount will then be compared across these segments.

The objective is to determine whether observed default risk differs across borrower income groups.

In [25]:
credit_analysis_df["income_segment"] = pd.qcut(
    credit_analysis_df["person_income"],
    q=4,
    labels=[
        "Low Income",
        "Lower-Middle Income",
        "Upper-Middle Income",
        "High Income"
    ]
)

income_analysis = (
    credit_analysis_df
    .groupby(
        "income_segment",
        observed=True
    )
    .agg(
        total_loans=("loan_status", "count"),
        defaults=("loan_status", "sum"),
        default_rate=("loan_status", "mean"),
        avg_income=("person_income", "mean"),
        avg_loan_amount=("loan_amnt", "mean")
    )
    .reset_index()
)

income_analysis

,income_segment,total_loans,defaults,default_rate,avg_income,avg_loan_amount
0,Low Income,8152,3236,0.396958,28615.324092,6308.234176
1,Lower-Middle Income,8231,1755,0.213218,47041.932815,8499.614263
2,Upper-Middle Income,8055,1375,0.170701,66282.422595,10453.438858
3,High Income,8143,742,0.091121,122609.044947,13120.938229


## 7. Default Risk by Loan-to-Income Ratio

The loan-to-income ratio represents the relative size of the requested loan compared with the borrower's income.

A higher ratio may indicate greater repayment pressure because the loan represents a larger proportion of the borrower's income.

Borrowers will therefore be grouped into loan-to-income bands and their observed default rates will be compared.

In [26]:
credit_analysis_df["loan_income_segment"] = pd.cut(
    credit_analysis_df["loan_percent_income"],
    bins=[
        -np.inf,
        0.10,
        0.20,
        0.30,
        0.40,
        np.inf
    ],
    labels=[
        "<=10%",
        "10%-20%",
        "20%-30%",
        "30%-40%",
        ">40%"
    ]
)

loan_income_analysis = (
    credit_analysis_df
    .groupby(
        "loan_income_segment",
        observed=True
    )
    .agg(
        total_loans=("loan_status", "count"),
        defaults=("loan_status", "sum"),
        default_rate=("loan_status", "mean"),
        avg_loan_amount=("loan_amnt", "mean")
    )
    .reset_index()
)

loan_income_analysis

,loan_income_segment,total_loans,defaults,default_rate,avg_loan_amount
0,<=10%,10484,1229,0.117226,5253.381343
1,10%-20%,12066,1823,0.151086,9646.929388
2,20%-30%,6197,1360,0.219461,12978.864773
3,30%-40%,2714,1865,0.687178,15136.560427
4,>40%,1120,831,0.741964,17361.026786


## 8. Default Risk by Interest Rate

Interest rate is an important lending variable because it reflects the cost of borrowing and may also be associated with the risk profile assigned to a loan.

Loans will be grouped into interest-rate bands and their observed default rates will be compared.

The objective is to identify whether default risk varies across interest-rate segments.

In [27]:
credit_analysis_df["interest_rate_segment"] = pd.cut(
    credit_analysis_df["loan_int_rate"],
    bins=[
        -np.inf,
        8,
        12,
        16,
        20,
        np.inf
    ],
    labels=[
        "<8%",
        "8%-12%",
        "12%-16%",
        "16%-20%",
        ">20%"
    ]
)

interest_analysis = (
    credit_analysis_df
    .groupby(
        "interest_rate_segment",
        observed=True
    )
    .agg(
        total_loans=("loan_status", "count"),
        defaults=("loan_status", "sum"),
        default_rate=("loan_status", "mean"),
        avg_loan_amount=("loan_amnt", "mean")
    )
    .reset_index()
)

interest_analysis

,interest_rate_segment,total_loans,defaults,default_rate,avg_loan_amount
0,<8%,7959,744,0.093479,8461.282824
1,8%-12%,10359,1638,0.158123,9663.879235
2,12%-16%,9373,2967,0.316548,9912.242612
3,16%-20%,1700,1052,0.618824,12340.294118
4,>20%,74,63,0.851351,14555.067568


## 9. Default Risk by Employment Length

Employment length can provide an indication of employment stability.

Borrowers will be grouped according to employment duration and the observed default rate will be compared across these groups.

This analysis helps determine whether shorter or longer employment histories are associated with different levels of observed loan risk.

In [28]:
credit_analysis_df["employment_segment"] = pd.cut(
    credit_analysis_df["person_emp_length"],
    bins=[
        -np.inf,
        2,
        5,
        10,
        np.inf
    ],
    labels=[
        "0-2 Years",
        "3-5 Years",
        "6-10 Years",
        "10+ Years"
    ]
)

employment_analysis = (
    credit_analysis_df
    .groupby(
        "employment_segment",
        observed=True
    )
    .agg(
        total_loans=("loan_status", "count"),
        defaults=("loan_status", "sum"),
        default_rate=("loan_status", "mean"),
        avg_income=("person_income", "mean")
    )
    .reset_index()
)

employment_analysis

,employment_segment,total_loans,defaults,default_rate,avg_income
0,0-2 Years,10869,2940,0.270494,59405.530960
1,3-5 Years,9276,1854,0.199871,64165.726822
2,6-10 Years,8612,1556,0.180678,70928.084649
3,10+ Years,2929,476,0.162513,89274.903721


## 10. Default Risk by Credit History Length

Credit history length represents the amount of time for which a borrower has established credit history.

The analysis will compare default rates across different credit-history segments.

This can help determine whether observed loan performance differs according to the length of borrowers' credit histories.

In [29]:
credit_analysis_df["credit_history_segment"] = pd.cut(
    credit_analysis_df["cb_person_cred_hist_length"],
    bins=[
        -np.inf,
        3,
        5,
        10,
        np.inf
    ],
    labels=[
        "<=3 Years",
        "4-5 Years",
        "6-10 Years",
        "10+ Years"
    ]
)

credit_history_analysis = (
    credit_analysis_df
    .groupby(
        "credit_history_segment",
        observed=True
    )
    .agg(
        total_loans=("loan_status", "count"),
        defaults=("loan_status", "sum"),
        default_rate=("loan_status", "mean"),
        avg_loan_amount=("loan_amnt", "mean")
    )
    .reset_index()
)

credit_history_analysis

,credit_history_segment,total_loans,defaults,default_rate,avg_loan_amount
0,<=3 Years,11908,2730,0.229258,9276.186177
1,4-5 Years,7806,1710,0.219062,9478.241097
2,6-10 Years,9405,1941,0.206380,9931.350346
3,10+ Years,3462,727,0.209994,9988.149913


## 11. Default Risk by Previous Credit Default

The `cb_person_default_on_file` variable indicates whether a borrower has a recorded previous credit default.

Current loan outcomes will be compared between borrowers with and without a previous default record.

The objective is to determine whether previous credit behavior is associated with current loan default risk.

In [30]:
previous_default_analysis = (
    credit_analysis_df
    .groupby("cb_person_default_on_file")
    .agg(
        total_loans=("loan_status", "count"),
        defaults=("loan_status", "sum"),
        default_rate=("loan_status", "mean"),
        avg_loan_amount=("loan_amnt", "mean"),
        avg_income=("person_income", "mean")
    )
    .reset_index()
)

previous_default_analysis

,cb_person_default_on_file,total_loans,defaults,default_rate,avg_loan_amount,avg_income
0,N,26836,4936,0.183932,9475.055895,66178.476263
1,Y,5745,2172,0.378068,10123.359443,65590.783116


## 12. Customer Risk Segmentation

A simple rule-based risk score will be created using several observed risk indicators:

- Higher-risk loan grades
- Higher loan-to-income ratio
- Higher interest rates
- Previous credit default

Each indicator contributes one point to the risk score.

The resulting score will be used to create Low, Moderate, High, and Very High Risk segments.

This segmentation is intended for business analysis and interpretability. It is not a replacement for the machine-learning model.

In [31]:
credit_analysis_df["risk_score"] = 0

credit_analysis_df.loc[
    credit_analysis_df["loan_grade"].isin(
        ["D", "E", "F", "G"]
    ),
    "risk_score"
] += 1

credit_analysis_df.loc[
    credit_analysis_df["loan_percent_income"] > 0.30,
    "risk_score"
] += 1

credit_analysis_df.loc[
    credit_analysis_df["loan_int_rate"] > 16,
    "risk_score"
] += 1

credit_analysis_df.loc[
    credit_analysis_df["cb_person_default_on_file"] == "Y",
    "risk_score"
] += 1

In [32]:
credit_analysis_df["risk_segment"] = pd.cut(
    credit_analysis_df["risk_score"],
    bins=[
        -1,
        0,
        1,
        2,
        4
    ],
    labels=[
        "Low Risk",
        "Moderate Risk",
        "High Risk",
        "Very High Risk"
    ]
)

risk_segment_analysis = (
    credit_analysis_df
    .groupby(
        "risk_segment",
        observed=True
    )
    .agg(
        total_loans=("loan_status", "count"),
        defaults=("loan_status", "sum"),
        default_rate=("loan_status", "mean"),
        avg_loan_amount=("loan_amnt", "mean"),
        avg_income=("person_income", "mean"),
        avg_interest_rate=("loan_int_rate", "mean")
    )
    .reset_index()
)

risk_segment_analysis

,risk_segment,total_loans,defaults,default_rate,avg_loan_amount,avg_income,avg_interest_rate
0,Low Risk,21749,1693,0.077843,8534.696998,69444.581452,9.640308
1,Moderate Risk,6842,2859,0.417860,11315.777550,57142.642064,12.412833
2,High Risk,2695,1661,0.616327,11453.756957,63441.345826,15.448134
3,Very High Risk,1295,895,0.691120,14300.965251,62154.480309,16.983635


## 13. Portfolio Risk Concentration

Default rate alone does not fully describe portfolio risk.

A segment may have a high default rate but contain relatively few loans.

Therefore, portfolio analysis should also consider total loan exposure.

This analysis compares default rates with total loan exposure to identify segments that may contribute substantially to overall portfolio risk.

In [33]:
portfolio_risk = (
    credit_analysis_df
    .groupby(
        "risk_segment",
        observed=True
    )
    .agg(
        total_loans=("loan_status", "count"),
        defaults=("loan_status", "sum"),
        total_exposure=("loan_amnt", "sum"),
        avg_loan_amount=("loan_amnt", "mean")
    )
    .reset_index()
)

portfolio_risk["default_rate"] = (
    portfolio_risk["defaults"]
    / portfolio_risk["total_loans"]
)

portfolio_risk["exposure_share"] = (
    portfolio_risk["total_exposure"]
    / portfolio_risk["total_exposure"].sum()
)

portfolio_risk

,risk_segment,total_loans,defaults,total_exposure,avg_loan_amount,default_rate,exposure_share
0,Low Risk,21749,1693,185621125,8534.696998,0.077843,0.594118
1,Moderate Risk,6842,2859,77422550,11315.777550,0.417860,0.247807
2,High Risk,2695,1661,30867875,11453.756957,0.616327,0.098799
3,Very High Risk,1295,895,18519750,14300.965251,0.691120,0.059276


In [34]:
risk_dashboard = portfolio_risk[
    [
        "risk_segment",
        "total_loans",
        "defaults",
        "default_rate",
        "total_exposure",
        "exposure_share"
    ]
].copy()

risk_dashboard

,risk_segment,total_loans,defaults,default_rate,total_exposure,exposure_share
0,Low Risk,21749,1693,0.077843,185621125,0.594118
1,Moderate Risk,6842,2859,0.417860,77422550,0.247807
2,High Risk,2695,1661,0.616327,30867875,0.098799
3,Very High Risk,1295,895,0.691120,18519750,0.059276


In [35]:
risk_dashboard["default_rate_pct"] = (
    risk_dashboard["default_rate"] * 100
)

risk_dashboard["exposure_share_pct"] = (
    risk_dashboard["exposure_share"] * 100
)

risk_dashboard

,risk_segment,total_loans,defaults,default_rate,total_exposure,exposure_share,default_rate_pct,exposure_share_pct
0,Low Risk,21749,1693,0.077843,185621125,0.594118,7.784266,59.411821
1,Moderate Risk,6842,2859,0.417860,77422550,0.247807,41.786027,24.780664
2,High Risk,2695,1661,0.616327,30867875,0.098799,61.632653,9.879892
3,Very High Risk,1295,895,0.691120,18519750,0.059276,69.111969,5.927623


## 13.1 Business Risk Segmentation Findings

The business segmentation analysis provides a portfolio-level view of observed credit risk.

The analysis considers multiple borrower and loan characteristics, including loan grade, loan-to-income ratio, interest rate, and previous credit default.

The resulting risk segments allow the portfolio to be examined according to both observed default rates and loan exposure.

This distinction is important because a segment with a high default rate does not necessarily represent the largest financial exposure.

From a business perspective, higher-risk segments may warrant:

- Enhanced underwriting
- Additional documentation
- Risk-based pricing
- Lower lending limits
- Increased post-disbursement monitoring

The segmentation is an analytical framework and should not be interpreted as a causal or regulatory credit-scoring system.

The machine-learning models developed earlier provide a more flexible approach for predicting individual loan-default risk.

# 14. SQL Business Analytics

The previous sections performed business analysis using Pandas.

In this section, the same business questions will be answered using actual SQL queries.

SQLite will be used as the analytical database because it is lightweight and does not require a separate database server.

The SQL analysis will cover:

- Filtering
- Aggregation
- GROUP BY
- HAVING
- CASE WHEN
- Subqueries
- Common Table Expressions (CTEs)
- Window functions
- Ranking
- Business KPIs
- Credit-risk analysis
- Fraud analysis

The objective is to demonstrate practical SQL skills that can be applied to banking and financial analytics.

In [36]:
import sqlite3

In [37]:
sql_connection = sqlite3.connect(
    "../data/banking_analytics.db"
)

print("SQLite database connected successfully.")

SQLite database connected successfully.


## 14.1 Create Credit Risk SQL Table

The Credit Risk dataframe will be loaded into a SQLite table called `credit_risk`.

This table will serve as the primary source for the SQL-based lending analysis.

In [38]:
credit_df.to_sql(
    "credit_risk",
    sql_connection,
    if_exists="replace",
    index=False
)

print(
    "Credit Risk table created successfully."
)

Credit Risk table created successfully.


In [39]:
tables_query = """
SELECT name
FROM sqlite_master
WHERE type = 'table';
"""

pd.read_sql_query(
    tables_query,
    sql_connection
)

,name
0,credit_risk


## 15. Basic SQL Query

The first query retrieves a sample of records from the credit-risk table.

This demonstrates the fundamental `SELECT` and `LIMIT` operations.

## 15.1 Customer and Loan Information

Instead of retrieving every column, specific business-relevant columns can be selected.

This is useful when analysts need only the information required for a particular analysis.

## 15.2 Filtering High-Value Loans

The `WHERE` clause is used to filter records based on business conditions.

The following query identifies loans greater than $20,000.

In [40]:
query = """
SELECT
    person_age,
    person_income,
    loan_grade,
    loan_amnt,
    loan_status
FROM credit_risk
WHERE loan_amnt > 20000
LIMIT 20;
"""

pd.read_sql_query(
    query,
    sql_connection
)

,person_age,person_income,loan_grade,loan_amnt,loan_status
0,22,59000,D,35000,1
1,23,65500,C,35000,1
2,24,54400,C,35000,1
3,26,77100,B,35000,1
4,24,78956,B,35000,1
5,24,83000,A,35000,1
6,22,85000,B,35000,1
7,23,95000,A,35000,1
8,26,108160,E,35000,1
9,23,115000,A,35000,0


## 16. Default Rate by Loan Grade

`GROUP BY` is used to aggregate loan performance by loan grade.

The query calculates:

- Total loans
- Number of defaults
- Default rate

This reproduces one of the key business analyses previously performed using Pandas.

In [41]:
query = """
SELECT
    loan_grade,
    COUNT(*) AS total_loans,
    SUM(loan_status) AS defaults,
    ROUND(
        AVG(loan_status) * 100,
        2
    ) AS default_rate_pct
FROM credit_risk
GROUP BY loan_grade
ORDER BY default_rate_pct DESC;
"""

pd.read_sql_query(
    query,
    sql_connection
)

,loan_grade,total_loans,defaults,default_rate_pct
0,G,64,63,98.44
1,F,241,170,70.54
2,E,964,621,64.42
3,D,3626,2141,59.05
4,C,6458,1339,20.73
5,B,10451,1701,16.28
6,A,10777,1073,9.96


## 16.1 Default Rate by Loan Intent

The following query compares observed default rates across different loan purposes.

This helps identify loan-purpose segments with relatively higher observed default risk.

In [42]:
query = """
SELECT
    loan_intent,
    COUNT(*) AS total_loans,
    SUM(loan_status) AS defaults,
    ROUND(
        AVG(loan_status) * 100,
        2
    ) AS default_rate_pct,
    ROUND(
        AVG(loan_amnt),
        2
    ) AS avg_loan_amount
FROM credit_risk
GROUP BY loan_intent
ORDER BY default_rate_pct DESC;
"""

pd.read_sql_query(
    query,
    sql_connection
)

,loan_intent,total_loans,defaults,default_rate_pct,avg_loan_amount
0,DEBTCONSOLIDATION,5212,1490,28.59,9594.89
1,MEDICAL,6071,1621,26.70,9259.58
2,HOMEIMPROVEMENT,3605,941,26.10,10360.52
3,PERSONAL,5521,1098,19.89,9573.77
4,EDUCATION,6453,1111,17.22,9482.68
5,VENTURE,5719,847,14.81,9583.78


## 16.2 Identifying High-Risk Loan Grades

The `HAVING` clause filters aggregated groups after `GROUP BY`.

The following query identifies loan grades with an observed default rate above 50%.

In [43]:
query = """
SELECT
    loan_grade,
    COUNT(*) AS total_loans,
    SUM(loan_status) AS defaults,
    ROUND(
        AVG(loan_status) * 100,
        2
    ) AS default_rate_pct
FROM credit_risk
GROUP BY loan_grade
HAVING AVG(loan_status) > 0.50
ORDER BY default_rate_pct DESC;
"""

pd.read_sql_query(
    query,
    sql_connection
)

,loan_grade,total_loans,defaults,default_rate_pct
0,G,64,63,98.44
1,F,241,170,70.54
2,E,964,621,64.42
3,D,3626,2141,59.05


## 17. Customer Risk Segmentation Using CASE WHEN

SQL `CASE WHEN` logic can be used to create business-oriented risk categories directly inside a query.

Borrowers will be classified based on their loan-to-income ratio.

query = """
SELECT
    person_age,
    person_income,
    loan_amnt,
    loan_percent_income,

    CASE
        WHEN loan_percent_income <= 0.10
            THEN 'Low Loan Burden'

        WHEN loan_percent_income <= 0.20
            THEN 'Moderate Loan Burden'

        WHEN loan_percent_income <= 0.30
            THEN 'High Loan Burden'

        ELSE 'Very High Loan Burden'
    END AS loan_burden_segment,

    loan_status

FROM credit_risk
LIMIT 20;
"""

pd.read_sql_query(
    query,
    sql_connection
)

query = """
SELECT

    CASE
        WHEN loan_percent_income <= 0.10
            THEN 'Low Loan Burden'

        WHEN loan_percent_income <= 0.20
            THEN 'Moderate Loan Burden'

        WHEN loan_percent_income <= 0.30
            THEN 'High Loan Burden'

        ELSE 'Very High Loan Burden'
    END AS loan_burden_segment,

    COUNT(*) AS total_loans,

    SUM(loan_status) AS defaults,

    ROUND(
        AVG(loan_status) * 100,
        2
    ) AS default_rate_pct

FROM credit_risk

GROUP BY loan_burden_segment

ORDER BY default_rate_pct DESC;
"""

pd.read_sql_query(
    query,
    sql_connection
)

# 18. Advanced SQL Analysis

Subqueries allow one SQL query to use the result of another query.

In banking analytics, subqueries can be used to identify:

- Customers above portfolio averages
- Loans above average exposure
- Segments with unusually high default rates
- High-risk borrowers

## 18.1 Loans Above the Portfolio Average

Identify loans whose amount is greater than the average loan amount across the entire portfolio.

This demonstrates the use of a scalar subquery.

In [44]:
query = """
SELECT
    person_age,
    person_income,
    loan_grade,
    loan_amnt,
    loan_status
FROM credit_risk
WHERE loan_amnt > (
    SELECT AVG(loan_amnt)
    FROM credit_risk
)
ORDER BY loan_amnt DESC
LIMIT 20;
"""

pd.read_sql_query(
    query,
    sql_connection
)

,person_age,person_income,loan_grade,loan_amnt,loan_status
0,22,59000,D,35000,1
1,23,65500,C,35000,1
2,24,54400,C,35000,1
3,26,77100,B,35000,1
4,24,78956,B,35000,1
5,24,83000,A,35000,1
6,22,85000,B,35000,1
7,23,95000,A,35000,1
8,26,108160,E,35000,1
9,23,115000,A,35000,0


In [45]:
query = """
SELECT
    person_age,
    person_income,
    loan_amnt,
    loan_grade,
    loan_status
FROM credit_risk
WHERE person_income > (
    SELECT AVG(person_income)
    FROM credit_risk
)
ORDER BY person_income DESC
LIMIT 20;
"""

pd.read_sql_query(
    query,
    sql_connection
)

,person_age,person_income,loan_amnt,loan_grade,loan_status
0,144,6000000,5000,C,0
1,42,2039784,8450,C,0
2,60,1900000,1500,A,0
3,63,1782000,12025,C,0
4,44,1440000,6400,A,0
5,47,1362000,6600,A,0
6,32,1200000,12000,A,0
7,36,1200000,10000,A,0
8,40,1200000,10000,A,0
9,34,948000,2000,B,0


# 19. Common Table Expressions (CTEs)

A Common Table Expression (CTE) temporarily stores the result of a query that can then be referenced by a subsequent query.

CTEs improve readability and are particularly useful for multi-step analytical problems.

The following analysis first calculates default rates by loan grade and then identifies grades with above-average default rates.

In [46]:
query = """
WITH grade_risk AS (

    SELECT
        loan_grade,
        COUNT(*) AS total_loans,
        SUM(loan_status) AS defaults,
        AVG(loan_status) AS default_rate
    FROM credit_risk
    GROUP BY loan_grade

)

SELECT
    loan_grade,
    total_loans,
    defaults,
    ROUND(default_rate * 100, 2) AS default_rate_pct

FROM grade_risk

WHERE default_rate > (
    SELECT AVG(default_rate)
    FROM grade_risk
)

ORDER BY default_rate DESC;
"""

pd.read_sql_query(
    query,
    sql_connection
)

,loan_grade,total_loans,defaults,default_rate_pct
0,G,64,63,98.44
1,F,241,170,70.54
2,E,964,621,64.42
3,D,3626,2141,59.05


## 19.1 High-Risk Loan Intent Segments

A CTE will be used to calculate default rates by loan intent and identify loan purposes with above-average observed default rates.

In [47]:
query = """
WITH intent_risk AS (

    SELECT
        loan_intent,
        COUNT(*) AS total_loans,
        SUM(loan_status) AS defaults,
        AVG(loan_status) AS default_rate
    FROM credit_risk
    GROUP BY loan_intent

)

SELECT
    loan_intent,
    total_loans,
    defaults,
    ROUND(default_rate * 100, 2) AS default_rate_pct

FROM intent_risk

WHERE default_rate > (
    SELECT AVG(default_rate)
    FROM intent_risk
)

ORDER BY default_rate DESC;
"""

pd.read_sql_query(
    query,
    sql_connection
)

,loan_intent,total_loans,defaults,default_rate_pct
0,DEBTCONSOLIDATION,5212,1490,28.59
1,MEDICAL,6071,1621,26.70
2,HOMEIMPROVEMENT,3605,941,26.10


# 20. SQL Window Functions

Window functions perform calculations across related rows without collapsing the result into a single row per group.

They are commonly used for:

- Ranking
- Running totals
- Percentage calculations
- Comparisons with group averages
- Portfolio analysis

The `RANK()` function will be used to rank loan grades according to observed default rate.

In [48]:
query = """
WITH grade_risk AS (

    SELECT
        loan_grade,
        COUNT(*) AS total_loans,
        SUM(loan_status) AS defaults,
        AVG(loan_status) AS default_rate
    FROM credit_risk
    GROUP BY loan_grade

)

SELECT
    loan_grade,
    total_loans,
    defaults,
    ROUND(default_rate * 100, 2) AS default_rate_pct,

    RANK() OVER (
        ORDER BY default_rate DESC
    ) AS risk_rank

FROM grade_risk

ORDER BY risk_rank;
"""

pd.read_sql_query(
    query,
    sql_connection
)

,loan_grade,total_loans,defaults,default_rate_pct,risk_rank
0,G,64,63,98.44,1
1,F,241,170,70.54,2
2,E,964,621,64.42,3
3,D,3626,2141,59.05,4
4,C,6458,1339,20.73,5
5,B,10451,1701,16.28,6
6,A,10777,1073,9.96,7


## 20.1 Ranking Loan Grades

`DENSE_RANK()` assigns rankings without gaps when two or more rows have the same value.

It is useful when ranking customer or portfolio segments by a business KPI.

In [49]:
query = """
WITH grade_risk AS (

    SELECT
        loan_grade,
        AVG(loan_status) AS default_rate
    FROM credit_risk
    GROUP BY loan_grade

)

SELECT
    loan_grade,

    ROUND(
        default_rate * 100,
        2
    ) AS default_rate_pct,

    DENSE_RANK() OVER (
        ORDER BY default_rate DESC
    ) AS risk_rank

FROM grade_risk

ORDER BY risk_rank;
"""

pd.read_sql_query(
    query,
    sql_connection
)

,loan_grade,default_rate_pct,risk_rank
0,G,98.44,1
1,F,70.54,2
2,E,64.42,3
3,D,59.05,4
4,C,20.73,5
5,B,16.28,6
6,A,9.96,7


## 21. Portfolio Exposure by Loan Grade

Default rate does not represent the complete financial risk of a portfolio.

This analysis calculates total loan exposure by grade and ranks each grade according to its total exposure.

This allows risk managers to distinguish between:

- High default rate
- High financial exposure
- High default rate and high exposure

In [50]:
query = """
WITH grade_exposure AS (

    SELECT
        loan_grade,
        COUNT(*) AS total_loans,
        SUM(loan_amnt) AS total_exposure,
        AVG(loan_status) AS default_rate
    FROM credit_risk
    GROUP BY loan_grade

)

SELECT
    loan_grade,
    total_loans,
    total_exposure,

    ROUND(
        default_rate * 100,
        2
    ) AS default_rate_pct,

    RANK() OVER (
        ORDER BY total_exposure DESC
    ) AS exposure_rank

FROM grade_exposure

ORDER BY exposure_rank;
"""

pd.read_sql_query(
    query,
    sql_connection
)

,loan_grade,total_loans,total_exposure,default_rate_pct,exposure_rank
0,B,10451,104462800,16.28,1
1,A,10777,92027750,9.96,2
2,C,6458,59503125,20.73,3
3,D,3626,39339350,59.05,4
4,E,964,12450875,64.42,5
5,F,241,3546875,70.54,6
6,G,64,1100525,98.44,7


## 21.1 Running Portfolio Exposure

A running total can show how portfolio exposure accumulates across loan grades.

The `SUM() OVER()` window function will be used to calculate cumulative loan exposure.

In [51]:
query = """
WITH grade_exposure AS (

    SELECT
        loan_grade,
        SUM(loan_amnt) AS total_exposure
    FROM credit_risk
    GROUP BY loan_grade

)

SELECT
    loan_grade,
    total_exposure,

    SUM(total_exposure) OVER (
        ORDER BY loan_grade
        ROWS BETWEEN UNBOUNDED PRECEDING
        AND CURRENT ROW
    ) AS cumulative_exposure

FROM grade_exposure

ORDER BY loan_grade;
"""

pd.read_sql_query(
    query,
    sql_connection
)

,loan_grade,total_exposure,cumulative_exposure
0,A,92027750,92027750
1,B,104462800,196490550
2,C,59503125,255993675
3,D,39339350,295333025
4,E,12450875,307783900
5,F,3546875,311330775
6,G,1100525,312431300


## 21.2 Segment Default Rate vs Portfolio Default Rate

The query compares each loan grade's observed default rate with the overall portfolio default rate.

This identifies segments performing above or below the portfolio baseline.

In [52]:
query = """
WITH grade_risk AS (

    SELECT
        loan_grade,
        AVG(loan_status) AS default_rate
    FROM credit_risk
    GROUP BY loan_grade

),

portfolio AS (

    SELECT
        AVG(loan_status) AS portfolio_default_rate
    FROM credit_risk

)

SELECT
    g.loan_grade,

    ROUND(
        g.default_rate * 100,
        2
    ) AS grade_default_rate_pct,

    ROUND(
        p.portfolio_default_rate * 100,
        2
    ) AS portfolio_default_rate_pct,

    ROUND(
        (g.default_rate - p.portfolio_default_rate) * 100,
        2
    ) AS difference_from_portfolio_pct

FROM grade_risk g
CROSS JOIN portfolio p

ORDER BY difference_from_portfolio_pct DESC;
"""

pd.read_sql_query(
    query,
    sql_connection
)

,loan_grade,grade_default_rate_pct,portfolio_default_rate_pct,difference_from_portfolio_pct
0,G,98.44,21.82,76.62
1,F,70.54,21.82,48.72
2,E,64.42,21.82,42.60
3,D,59.05,21.82,37.23
4,C,20.73,21.82,-1.08
5,B,16.28,21.82,-5.54
6,A,9.96,21.82,-11.86


## 22. High-Risk Loan Identification

A practical lending analytics system may need to identify individual loans with multiple risk indicators.

The following query creates a simple rule-based risk score using:

- Loan grade
- Loan-to-income ratio
- Interest rate
- Previous credit default

This SQL-based score is intended for analytical demonstration and is not a replacement for the machine-learning model.

In [53]:
query = """
SELECT
    person_age,
    person_income,
    loan_grade,
    loan_amnt,
    loan_int_rate,
    loan_percent_income,
    cb_person_default_on_file,
    loan_status,

    (
        CASE
            WHEN loan_grade IN ('D', 'E', 'F', 'G')
            THEN 1
            ELSE 0
        END

        +

        CASE
            WHEN loan_percent_income > 0.30
            THEN 1
            ELSE 0
        END

        +

        CASE
            WHEN loan_int_rate > 16
            THEN 1
            ELSE 0
        END

        +

        CASE
            WHEN cb_person_default_on_file = 'Y'
            THEN 1
            ELSE 0
        END
    ) AS risk_score

FROM credit_risk

ORDER BY risk_score DESC

LIMIT 20;
"""

pd.read_sql_query(
    query,
    sql_connection
)

,person_age,person_income,loan_grade,loan_amnt,loan_int_rate,loan_percent_income,cb_person_default_on_file,loan_status,risk_score
0,22,59000,D,35000,16.02,0.59,Y,1,4
1,21,11000,E,4575,17.74,0.42,Y,1,4
2,25,75000,D,30000,16.89,0.40,Y,1,4
3,23,78000,F,30000,18.62,0.38,Y,1,4
4,25,42360,E,25000,16.35,0.59,Y,1,4
5,23,78000,D,25000,16.32,0.32,Y,1,4
6,25,66300,E,25000,17.93,0.32,Y,1,4
7,26,78000,E,24250,17.56,0.31,Y,1,4
8,24,48000,D,24000,16.29,0.50,Y,1,4
9,21,44964,D,24000,16.29,0.53,Y,1,4


# 23. Fraud Detection SQL Analytics

The fraud dataset will now be added to the SQLite database.

The objective is to use SQL to investigate fraud patterns and generate business-oriented insights.

Because fraudulent transactions are extremely rare, the analysis will focus on:

- Fraud rate
- Fraud transaction counts
- Transaction amount
- Fraud exposure
- Feature-level fraud patterns
- High-value fraudulent transactions
- Comparative fraud metrics

The analysis will complement the machine-learning fraud-detection workflow developed earlier.

In [54]:
fraud_sql_df = pd.read_csv(
    r"D:\Banking-Credit-Risk-Fraud-Analytics\data\fraud\creditcard.csv"
)

print(
    "Fraud dataset shape:",
    fraud_sql_df.shape
)

print(
    "\nColumns:"
)

print(
    fraud_sql_df.columns.tolist()
)

Fraud dataset shape: (284807, 31)

Columns:
['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']


## 23.1 Fraud Dataset Validation

Before loading the fraud dataset into SQLite, its structure and target distribution will be validated.

The `Class` variable represents the transaction outcome:

- `0` = Legitimate transaction
- `1` = Fraudulent transaction

In [55]:
print(
    fraud_sql_df.head()
)

print(
    "\nMissing values:"
)

print(
    fraud_sql_df.isnull().sum().sum()
)

print(
    "\nDuplicate rows:"
)

print(
    fraud_sql_df.duplicated().sum()
)

print(
    "\nClass distribution:"
)

print(
    fraud_sql_df["Class"].value_counts()
)

   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.141267 -0.206010   

        V26       V27       V28 

In [56]:
print(
    fraud_sql_df.head()
)

print(
    "\nMissing values:"
)

print(
    fraud_sql_df.isnull().sum().sum()
)

print(
    "\nDuplicate rows:"
)

print(
    fraud_sql_df.duplicated().sum()
)

print(
    "\nClass distribution:"
)

print(
    fraud_sql_df["Class"].value_counts()
)

   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.141267 -0.206010   

        V26       V27       V28 

In [57]:
fraud_sql_df.to_sql(
    "fraud_transactions",
    sql_connection,
    if_exists="replace",
    index=False
)

print(
    "Fraud transactions table created successfully."
)

Fraud transactions table created successfully.


In [58]:
query = """
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
"""

pd.read_sql_query(
    query,
    sql_connection
)

,name
0,credit_risk
1,fraud_transactions


## 24. Overall Fraud Rate

The first fraud SQL analysis calculates the total number of transactions, fraudulent transactions, legitimate transactions, and overall fraud rate.

This establishes the baseline for understanding the extreme class imbalance.

In [59]:
query = """
SELECT

    COUNT(*) AS total_transactions,

    SUM(
        CASE
            WHEN Class = 1 THEN 1
            ELSE 0
        END
    ) AS fraudulent_transactions,

    SUM(
        CASE
            WHEN Class = 0 THEN 1
            ELSE 0
        END
    ) AS legitimate_transactions,

    ROUND(
        AVG(Class) * 100,
        4
    ) AS fraud_rate_pct

FROM fraud_transactions;
"""

pd.read_sql_query(
    query,
    sql_connection
)

,total_transactions,fraudulent_transactions,legitimate_transactions,fraud_rate_pct
0,284807,492,284315,0.1727


## 24.1 Transaction Amount Comparison

Transaction amounts will be compared between legitimate and fraudulent transactions.

The analysis calculates:

- Transaction count
- Average amount
- Minimum amount
- Maximum amount
- Total transaction amount

This helps determine whether fraudulent transactions differ from legitimate transactions in terms of transaction value.

In [60]:
query = """
SELECT

    Class,

    COUNT(*) AS transaction_count,

    ROUND(
        AVG(Amount),
        2
    ) AS avg_amount,

    ROUND(
        MIN(Amount),
        2
    ) AS min_amount,

    ROUND(
        MAX(Amount),
        2
    ) AS max_amount,

    ROUND(
        SUM(Amount),
        2
    ) AS total_amount

FROM fraud_transactions

GROUP BY Class

ORDER BY Class;
"""

pd.read_sql_query(
    query,
    sql_connection
)

,Class,transaction_count,avg_amount,min_amount,max_amount,total_amount
0,0,284315,88.29,0.0,25691.16,25102462.04
1,1,492,122.21,0.0,2125.87,60127.97


## 25. High-Value Fraudulent Transactions

A fraud-monitoring team may prioritize high-value fraudulent transactions because the financial impact of each event can be larger.

The following query identifies the highest-value fraudulent transactions.

In [61]:
query = """
SELECT
    Time,
    Amount,
    Class
FROM fraud_transactions
WHERE Class = 1
ORDER BY Amount DESC
LIMIT 20;
"""

pd.read_sql_query(
    query,
    sql_connection
)

,Time,Amount,Class
0,122608.0,2125.87,1
1,9064.0,1809.68,1
2,154278.0,1504.93,1
3,62467.0,1402.16,1
4,59011.0,1389.56,1
5,65385.0,1354.25,1
6,133184.0,1335.00,1
7,18088.0,1218.89,1
8,154309.0,1096.99,1
9,147501.0,996.27,1


## 25.1 Fraudulent Transaction Exposure

Fraud count alone does not capture financial impact.

The total value of fraudulent transactions provides an estimate of the transaction-value exposure represented by the observed fraud cases.

This metric can help fraud teams prioritize financial impact alongside transaction volume.

In [62]:
query = """
SELECT

    COUNT(*) AS fraud_transactions,

    ROUND(
        SUM(Amount),
        2
    ) AS fraud_amount_exposure,

    ROUND(
        AVG(Amount),
        2
    ) AS average_fraud_amount,

    ROUND(
        MAX(Amount),
        2
    ) AS highest_fraud_amount

FROM fraud_transactions

WHERE Class = 1;
"""

pd.read_sql_query(
    query,
    sql_connection
)

,fraud_transactions,fraud_amount_exposure,average_fraud_amount,highest_fraud_amount
0,492,60127.97,122.21,2125.87


## 26. Fraud Rate by Transaction Amount Segment

Transactions will be divided into amount bands to examine whether observed fraud rates vary by transaction value.

The segments are:

- Under $50
- $50–$100
- $100–$500
- $500–$1,000
- Above $1,000

The analysis compares transaction volume, fraud count, and fraud rate across these segments.

In [63]:
query = """
SELECT

    CASE

        WHEN Amount < 50
            THEN '< $50'

        WHEN Amount < 100
            THEN '$50-$100'

        WHEN Amount < 500
            THEN '$100-$500'

        WHEN Amount < 1000
            THEN '$500-$1,000'

        ELSE '> $1,000'

    END AS amount_segment,

    COUNT(*) AS total_transactions,

    SUM(Class) AS fraud_transactions,

    ROUND(
        AVG(Class) * 100,
        4
    ) AS fraud_rate_pct,

    ROUND(
        SUM(Amount),
        2
    ) AS total_transaction_value

FROM fraud_transactions

GROUP BY amount_segment

ORDER BY fraud_rate_pct DESC;
"""

pd.read_sql_query(
    query,
    sql_connection
)

,amount_segment,total_transactions,fraud_transactions,fraud_rate_pct,total_transaction_value
0,"$500-$1,000",6423,26,0.4048,4346353.57
1,"> $1,000",3069,9,0.2933,5444309.98
2,$100-$500,47893,95,0.1984,9982528.36
3,< $50,189704,305,0.1608,2679438.23
4,$50-$100,37718,57,0.1511,2709959.87


## 26.1 Fraud Rate by Time Period

The `Time` variable represents elapsed time in seconds from the beginning of the observation period.

Transactions will be grouped into broad time intervals to explore whether fraudulent activity is concentrated during particular periods.

This analysis is exploratory because the dataset does not directly provide calendar timestamps.

In [64]:
query = """
SELECT

    CASE

        WHEN Time < 21600
            THEN '0-6 Hours'

        WHEN Time < 43200
            THEN '6-12 Hours'

        WHEN Time < 64800
            THEN '12-18 Hours'

        ELSE '18-24 Hours'

    END AS time_segment,

    COUNT(*) AS total_transactions,

    SUM(Class) AS fraud_transactions,

    ROUND(
        AVG(Class) * 100,
        4
    ) AS fraud_rate_pct

FROM fraud_transactions

GROUP BY time_segment

ORDER BY time_segment;
"""

pd.read_sql_query(
    query,
    sql_connection
)

,time_segment,total_transactions,fraud_transactions,fraud_rate_pct
0,0-6 Hours,12340,55,0.4457
1,12-18 Hours,46850,71,0.1515
2,18-24 Hours,190556,275,0.1443
3,6-12 Hours,35061,91,0.2595


## 26.2 Fraud Segment Performance vs Overall Fraud Rate

The following analysis compares fraud rates across transaction segments against the overall portfolio fraud rate.

This helps identify segments that exhibit relatively higher or lower observed fraud rates.

In [65]:
query = """
WITH amount_segments AS (

    SELECT

        CASE

            WHEN Amount < 50
                THEN '< $50'

            WHEN Amount < 100
                THEN '$50-$100'

            WHEN Amount < 500
                THEN '$100-$500'

            WHEN Amount < 1000
                THEN '$500-$1,000'

            ELSE '> $1,000'

        END AS amount_segment,

        AVG(Class) AS fraud_rate

    FROM fraud_transactions

    GROUP BY amount_segment

),

overall AS (

    SELECT
        AVG(Class) AS overall_fraud_rate

    FROM fraud_transactions

)

SELECT

    a.amount_segment,

    ROUND(
        a.fraud_rate * 100,
        4
    ) AS segment_fraud_rate_pct,

    ROUND(
        o.overall_fraud_rate * 100,
        4
    ) AS overall_fraud_rate_pct,

    ROUND(
        (a.fraud_rate - o.overall_fraud_rate) * 100,
        4
    ) AS difference_from_overall_pct

FROM amount_segments a

CROSS JOIN overall o

ORDER BY difference_from_overall_pct DESC;
"""

pd.read_sql_query(
    query,
    sql_connection
)

,amount_segment,segment_fraud_rate_pct,overall_fraud_rate_pct,difference_from_overall_pct
0,"$500-$1,000",0.4048,0.1727,0.2320
1,"> $1,000",0.2933,0.1727,0.1205
2,$100-$500,0.1984,0.1727,0.0256
3,< $50,0.1608,0.1727,-0.0120
4,$50-$100,0.1511,0.1727,-0.0216


# 27. Advanced Fraud SQL Analysis

The previous SQL analysis examined fraud rates and transaction-value patterns.

This section moves from descriptive analysis toward fraud-risk prioritization.

The analysis will combine:

- Transaction amount
- Fraud probability/rate by amount segment
- Ranking
- CTEs
- Window functions

The objective is to identify transaction segments that deserve greater fraud-monitoring attention.

In [66]:
query = """
WITH amount_analysis AS (

    SELECT

        CASE
            WHEN Amount < 50
                THEN '< $50'

            WHEN Amount < 100
                THEN '$50-$100'

            WHEN Amount < 500
                THEN '$100-$500'

            WHEN Amount < 1000
                THEN '$500-$1,000'

            ELSE '> $1,000'
        END AS amount_segment,

        COUNT(*) AS total_transactions,

        SUM(Class) AS fraud_transactions,

        AVG(Class) AS fraud_rate,

        SUM(Amount) AS total_value

    FROM fraud_transactions

    GROUP BY amount_segment

)

SELECT

    amount_segment,

    total_transactions,

    fraud_transactions,

    ROUND(
        fraud_rate * 100,
        4
    ) AS fraud_rate_pct,

    ROUND(
        total_value,
        2
    ) AS total_value,

    RANK() OVER (
        ORDER BY fraud_rate DESC
    ) AS fraud_rate_rank

FROM amount_analysis

ORDER BY fraud_rate_rank;
"""

pd.read_sql_query(
    query,
    sql_connection
)

,amount_segment,total_transactions,fraud_transactions,fraud_rate_pct,total_value,fraud_rate_rank
0,"$500-$1,000",6423,26,0.4048,4346353.57,1
1,"> $1,000",3069,9,0.2933,5444309.98,2
2,$100-$500,47893,95,0.1984,9982528.36,3
3,< $50,189704,305,0.1608,2679438.23,4
4,$50-$100,37718,57,0.1511,2709959.87,5


## 27.1 Rank Fraudulent Transactions by Amount

The `ROW_NUMBER()` window function assigns a unique rank to fraudulent transactions based on transaction amount.

This can be useful for prioritizing high-value fraud cases for investigation.

In [67]:
query = """
SELECT

    Time,
    Amount,
    Class,

    ROW_NUMBER() OVER (
        ORDER BY Amount DESC
    ) AS transaction_rank

FROM fraud_transactions

WHERE Class = 1

ORDER BY Amount DESC

LIMIT 20;
"""

pd.read_sql_query(
    query,
    sql_connection
)

,Time,Amount,Class,transaction_rank
0,122608.0,2125.87,1,1
1,9064.0,1809.68,1,2
2,154278.0,1504.93,1,3
3,62467.0,1402.16,1,4
4,59011.0,1389.56,1,5
5,65385.0,1354.25,1,6
6,133184.0,1335.00,1,7
7,18088.0,1218.89,1,8
8,154309.0,1096.99,1,9
9,147501.0,996.27,1,10


## 27.2 Fraud Transaction Amount vs Average Fraud Amount

A subquery can be used to compare each fraudulent transaction with the average fraudulent transaction amount.

This helps identify unusually high-value fraud events.

In [68]:
query = """
SELECT

    Time,
    Amount,

    ROUND(
        Amount - (
            SELECT AVG(Amount)
            FROM fraud_transactions
            WHERE Class = 1
        ),
        2
    ) AS difference_from_avg_fraud_amount

FROM fraud_transactions

WHERE Class = 1

ORDER BY difference_from_avg_fraud_amount DESC

LIMIT 20;
"""

pd.read_sql_query(
    query,
    sql_connection
)

,Time,Amount,difference_from_avg_fraud_amount
0,122608.0,2125.87,2003.66
1,9064.0,1809.68,1687.47
2,154278.0,1504.93,1382.72
3,62467.0,1402.16,1279.95
4,59011.0,1389.56,1267.35
5,65385.0,1354.25,1232.04
6,133184.0,1335.00,1212.79
7,18088.0,1218.89,1096.68
8,154309.0,1096.99,974.78
9,147501.0,996.27,874.06


## 28. High-Value Fraud Prioritization

Fraud investigations can be prioritized using transaction value.

The following query identifies fraudulent transactions whose amount is above the average fraud transaction amount.

These transactions may represent higher financial exposure and therefore may warrant additional investigation priority.

In [69]:
query = """
SELECT

    Time,
    Amount,
    Class

FROM fraud_transactions

WHERE Class = 1

AND Amount > (
    SELECT AVG(Amount)
    FROM fraud_transactions
    WHERE Class = 1
)

ORDER BY Amount DESC;
"""

pd.read_sql_query(
    query,
    sql_connection
)

,Time,Amount,Class
0,122608.0,2125.87,1
1,9064.0,1809.68,1
2,154278.0,1504.93,1
3,62467.0,1402.16,1
4,59011.0,1389.56,1
...,...,...,...
105,77154.0,129.00,1
106,160895.0,127.14,1
107,34684.0,125.30,1
108,75581.0,124.53,1


## 28.1 Fraud Exposure Concentration

The following analysis calculates what percentage of total observed fraudulent transaction value is represented by the highest-value fraudulent transactions.

This provides a business-oriented view of fraud concentration.

A small number of high-value transactions may account for a disproportionate share of total fraud exposure.

In [70]:
query = """
WITH ranked_fraud AS (

    SELECT

        Time,
        Amount,

        ROW_NUMBER() OVER (
            ORDER BY Amount DESC
        ) AS fraud_rank

    FROM fraud_transactions

    WHERE Class = 1

),

fraud_total AS (

    SELECT
        SUM(Amount) AS total_fraud_value
    FROM fraud_transactions
    WHERE Class = 1

)

SELECT

    r.fraud_rank,

    r.Time,

    r.Amount,

    ROUND(
        r.Amount / f.total_fraud_value * 100,
        4
    ) AS percentage_of_total_fraud_value

FROM ranked_fraud r

CROSS JOIN fraud_total f

ORDER BY r.fraud_rank

LIMIT 20;
"""

pd.read_sql_query(
    query,
    sql_connection
)

,fraud_rank,Time,Amount,percentage_of_total_fraud_value
0,1,122608.0,2125.87,3.5356
1,2,9064.0,1809.68,3.0097
2,3,154278.0,1504.93,2.5029
3,4,62467.0,1402.16,2.3320
4,5,59011.0,1389.56,2.3110
5,6,65385.0,1354.25,2.2523
6,7,133184.0,1335.00,2.2203
7,8,18088.0,1218.89,2.0272
8,9,154309.0,1096.99,1.8244
9,10,147501.0,996.27,1.6569


## 28.2 Cumulative Fraud Exposure

A cumulative sum is used to determine how quickly fraudulent transaction value accumulates when transactions are ordered from highest to lowest value.

This can help answer questions such as:

**"How much of the total fraud exposure comes from the top N highest-value fraudulent transactions?"**

In [71]:
query = """
WITH ranked_fraud AS (

    SELECT

        Time,
        Amount,

        ROW_NUMBER() OVER (
            ORDER BY Amount DESC
        ) AS fraud_rank

    FROM fraud_transactions

    WHERE Class = 1

),

fraud_total AS (

    SELECT
        SUM(Amount) AS total_fraud_value
    FROM fraud_transactions
    WHERE Class = 1

)

SELECT

    r.fraud_rank,

    r.Time,

    r.Amount,

    SUM(r.Amount) OVER (
        ORDER BY r.fraud_rank
        ROWS BETWEEN UNBOUNDED PRECEDING
        AND CURRENT ROW
    ) AS cumulative_fraud_value,

    ROUND(
        SUM(r.Amount) OVER (
            ORDER BY r.fraud_rank
            ROWS BETWEEN UNBOUNDED PRECEDING
            AND CURRENT ROW
        )
        / f.total_fraud_value * 100,
        2
    ) AS cumulative_exposure_pct

FROM ranked_fraud r

CROSS JOIN fraud_total f

ORDER BY r.fraud_rank

LIMIT 20;
"""

pd.read_sql_query(
    query,
    sql_connection
)

,fraud_rank,Time,Amount,cumulative_fraud_value,cumulative_exposure_pct
0,1,122608.0,2125.87,2125.87,3.54
1,2,9064.0,1809.68,3935.55,6.55
2,3,154278.0,1504.93,5440.48,9.05
3,4,62467.0,1402.16,6842.64,11.38
4,5,59011.0,1389.56,8232.20,13.69
5,6,65385.0,1354.25,9586.45,15.94
6,7,133184.0,1335.00,10921.45,18.16
7,8,18088.0,1218.89,12140.34,20.19
8,9,154309.0,1096.99,13237.33,22.02
9,10,147501.0,996.27,14233.60,23.67


## 29. Fraud vs Legitimate Transaction Statistics

A comparative analysis will examine the distribution of transaction values between legitimate and fraudulent transactions.

The analysis compares:

- Transaction count
- Average amount
- Total amount
- Maximum amount
- Minimum amount

This provides business context for the transaction-level fraud model.

In [72]:
query = """
SELECT

    CASE
        WHEN Class = 1
            THEN 'Fraud'
        ELSE 'Legitimate'
    END AS transaction_type,

    COUNT(*) AS transaction_count,

    ROUND(
        AVG(Amount),
        2
    ) AS avg_amount,

    ROUND(
        MIN(Amount),
        2
    ) AS min_amount,

    ROUND(
        MAX(Amount),
        2
    ) AS max_amount,

    ROUND(
        SUM(Amount),
        2
    ) AS total_amount

FROM fraud_transactions

GROUP BY Class;
"""

pd.read_sql_query(
    query,
    sql_connection
)

,transaction_type,transaction_count,avg_amount,min_amount,max_amount,total_amount
0,Legitimate,284315,88.29,0.0,25691.16,25102462.04
1,Fraud,492,122.21,0.0,2125.87,60127.97


## 30. Banking Risk KPI Dashboard

The following query combines multiple portfolio-level indicators into a single result.

The objective is to create a compact KPI layer that can later feed a dashboard or reporting system.

In [73]:
query = """
SELECT

    COUNT(*) AS total_loans,

    SUM(loan_status) AS total_defaults,

    ROUND(
        AVG(loan_status) * 100,
        2
    ) AS default_rate_pct,

    ROUND(
        SUM(loan_amnt),
        2
    ) AS total_loan_exposure,

    ROUND(
        AVG(loan_amnt),
        2
    ) AS average_loan_amount,

    ROUND(
        AVG(loan_int_rate),
        2
    ) AS average_interest_rate

FROM credit_risk;
"""

banking_kpis = pd.read_sql_query(
    query,
    sql_connection
)

banking_kpis

,total_loans,total_defaults,default_rate_pct,total_loan_exposure,average_loan_amount,average_interest_rate
0,32581,7108,21.82,312431300.0,9589.37,11.01


In [74]:
query = """
SELECT

    COUNT(*) AS total_transactions,

    SUM(
        CASE
            WHEN Class = 1 THEN 1
            ELSE 0
        END
    ) AS fraud_transactions,

    ROUND(
        AVG(Class) * 100,
        4
    ) AS fraud_rate_pct,

    ROUND(
        SUM(
            CASE
                WHEN Class = 1
                THEN Amount
                ELSE 0
            END
        ),
        2
    ) AS fraud_exposure,

    ROUND(
        AVG(
            CASE
                WHEN Class = 1
                THEN Amount
            END
        ),
        2
    ) AS avg_fraud_amount,

    ROUND(
        MAX(
            CASE
                WHEN Class = 1
                THEN Amount
            END
        ),
        2
    ) AS highest_fraud_amount

FROM fraud_transactions;
"""

fraud_kpis = pd.read_sql_query(
    query,
    sql_connection
)

fraud_kpis

,total_transactions,fraud_transactions,fraud_rate_pct,fraud_exposure,avg_fraud_amount,highest_fraud_amount
0,284807,492,0.1727,60127.97,122.21,2125.87


# 31. SQL Business Analytics — Key Findings

The SQL analysis translated the machine-learning datasets into business-oriented portfolio and fraud metrics.

## Credit Risk

The analysis demonstrated that:

- Default rates vary substantially across loan grades.
- Loan purpose is associated with different observed default rates.
- Customer housing status shows differences in observed loan performance.
- Loan burden and interest-rate segments can be used for portfolio-risk segmentation.
- Portfolio exposure should be evaluated alongside default rates.

## Fraud Detection

The analysis demonstrated that:

- Fraud represents an extremely small proportion of total transactions.
- Fraud transaction value can be analyzed separately from fraud transaction count.
- High-value fraudulent transactions can be prioritized for investigation.
- Fraud exposure can be ranked and accumulated using SQL window functions.
- Transaction-value segments can be compared by observed fraud rate.

## SQL Techniques Demonstrated

The project implemented practical SQL techniques including:

- SELECT
- WHERE
- GROUP BY
- HAVING
- CASE WHEN
- Aggregate functions
- Subqueries
- Common Table Expressions
- RANK
- DENSE_RANK
- ROW_NUMBER
- Window functions
- Running totals
- CROSS JOIN
- Business KPI calculations

These techniques demonstrate the ability to move beyond basic SQL filtering and perform analytical queries relevant to banking and financial datasets.

# 32. SQL Interview Preparation

This section converts the SQL analysis performed in this project into practical interview questions.

Each question connects a standard SQL concept with a real banking or fraud-analysis use case.

The objective is not only to write SQL but also to explain:

1. What the query does
2. Why the query is needed
3. What business problem it solves

## Q1. How would you calculate default rate by loan grade?

### Answer

Use `GROUP BY` to aggregate loans by `loan_grade`.

Because `loan_status` is coded as:

- 0 = Non-default
- 1 = Default

the average of `loan_status` gives the observed default rate.

### SQL

```sql
SELECT
    loan_grade,
    COUNT(*) AS total_loans,
    SUM(loan_status) AS defaults,
    AVG(loan_status) AS default_rate
FROM credit_risk
GROUP BY loan_grade
ORDER BY default_rate DESC;


---

# Question 2 — WHERE vs HAVING

### 📝 Markdown cell

```markdown id="v7m2q4"
## Q2. What is the difference between WHERE and HAVING?

### Answer

`WHERE` filters individual rows before aggregation.

`HAVING` filters aggregated groups after `GROUP BY`.

### Example

To identify loan grades with a default rate above 50%:

```sql
SELECT
    loan_grade,
    AVG(loan_status) AS default_rate
FROM credit_risk
GROUP BY loan_grade
HAVING AVG(loan_status) > 0.50;


---

# Question 3 — Subquery

### 📝 Markdown cell

```markdown id="r5m8q1"
## Q3. How would you identify loans above the portfolio's average loan amount?

### Answer

Use a scalar subquery to calculate the portfolio average and compare each loan against it.

```sql
SELECT
    person_age,
    loan_grade,
    loan_amnt,
    loan_status
FROM credit_risk
WHERE loan_amnt > (
    SELECT AVG(loan_amnt)
    FROM credit_risk
);


---

# Question 4 — CTE

### 📝 Markdown cell

```markdown id="q2m9x5"
## Q4. Why would you use a CTE?

### Answer

A Common Table Expression makes a complex query easier to read and maintain by breaking it into logical steps.

For example, we can first calculate default rate by loan grade and then rank the grades.

```sql
WITH grade_risk AS (

    SELECT
        loan_grade,
        AVG(loan_status) AS default_rate
    FROM credit_risk
    GROUP BY loan_grade

)

SELECT
    loan_grade,
    default_rate
FROM grade_risk
ORDER BY default_rate DESC;


---

# Question 5 — Window Function

### 📝 Markdown cell

```markdown id="k6m3q8"
## Q5. What is a window function?

### Answer

A window function performs a calculation across related rows while preserving the individual rows in the result.

For example:

```sql
RANK() OVER (
    ORDER BY default_rate DESC
)


---

# Question 6 — RANK vs DENSE_RANK

### 📝 Markdown cell

```markdown id="w4m8q2"
## Q6. What is the difference between RANK and DENSE_RANK?

### Answer

Both functions assign rankings.

The difference occurs when values are tied.

`RANK()` leaves gaps after ties.

`DENSE_RANK()` does not leave gaps.

Example:

Values:

10, 10, 8

RANK:

1, 1, 3

DENSE_RANK:

1, 1, 2

## Q7. How would you create risk categories in SQL?

### Answer

Use `CASE WHEN`.

```sql
SELECT

    person_income,

    loan_percent_income,

    CASE

        WHEN loan_percent_income <= 0.10
            THEN 'Low Loan Burden'

        WHEN loan_percent_income <= 0.20
            THEN 'Moderate Loan Burden'

        WHEN loan_percent_income <= 0.30
            THEN 'High Loan Burden'

        ELSE 'Very High Loan Burden'

    END AS loan_burden_segment

FROM credit_risk;


---

# Question 8 — Fraud Rate

### 📝 Markdown cell

```markdown id="m9q4x2"
## Q8. How would you calculate fraud rate in SQL?

### Answer

Because `Class = 1` represents fraud and `Class = 0` represents legitimate transactions, the average of `Class` gives the fraud rate.

```sql
SELECT
    COUNT(*) AS total_transactions,
    SUM(Class) AS fraud_transactions,
    AVG(Class) AS fraud_rate
FROM fraud_transactions;


---

# Question 9 — Top 3 Fraud Transactions

### 📝 Markdown cell

```markdown id="x3m7q5"
## Q9. How would you find the three highest-value fraudulent transactions?

### Answer

```sql
SELECT
    Time,
    Amount
FROM fraud_transactions
WHERE Class = 1
ORDER BY Amount DESC
LIMIT 3;


---

# Question 10 — Running Total

### 📝 Markdown cell

```markdown id="q8m4x1"
## Q10. How would you calculate cumulative fraud exposure?

### Answer

Use a windowed `SUM()`.

```sql
SELECT

    Time,
    Amount,

    SUM(Amount) OVER (
        ORDER BY Amount DESC
        ROWS BETWEEN UNBOUNDED PRECEDING
        AND CURRENT ROW
    ) AS cumulative_fraud_exposure

FROM fraud_transactions

WHERE Class = 1

ORDER BY Amount DESC;


---

# Step 411 — SQL + ML Interview Question

This is where you connect the entire project.

### 📝 Markdown cell

```markdown id="r7m3q8"
## Q11. Why isn't accuracy sufficient for fraud detection?

### Answer

Fraud detection has extreme class imbalance.

In this project, fraudulent transactions represent only approximately **0.173%** of all transactions.

A model could therefore achieve extremely high accuracy by predicting nearly every transaction as legitimate while failing to detect fraud.

For this reason, the project evaluates:

- Precision
- Recall
- F1-score
- ROC-AUC
- PR-AUC

PR-AUC is particularly important because it focuses on the precision-recall trade-off for the minority fraud class.

### Interview Explanation

"I would never select a fraud model based on accuracy alone. Because fraud is extremely rare, I would prioritize metrics such as recall, precision, F1 and especially PR-AUC, depending on the business cost of false positives versus missed fraud."

## Q12. How would you identify high-risk loan segments?

### Answer

I would combine multiple analytical indicators rather than relying on a single variable.

For example:

- Loan grade
- Loan-to-income ratio
- Interest rate
- Previous default history

I could use SQL `CASE WHEN` logic to create interpretable risk segments and then calculate default rates and loan exposure for each segment.

The machine-learning model would then be used for individual borrower-level prediction.

### Interview Explanation

"I would use SQL for portfolio-level segmentation and monitoring, while using the machine-learning model for more granular individual risk prediction."

# 33. SQL Skills Demonstrated

This project demonstrates practical SQL capabilities across credit-risk and fraud analytics.

## Core SQL

- SELECT
- WHERE
- GROUP BY
- ORDER BY
- LIMIT
- Aggregate functions

## Intermediate SQL

- HAVING
- CASE WHEN
- Subqueries
- Conditional aggregation

## Advanced SQL

- Common Table Expressions
- RANK
- DENSE_RANK
- ROW_NUMBER
- Window functions
- Running totals
- CROSS JOIN

## Business Applications

The SQL analysis was applied to:

- Loan default analysis
- Loan-grade risk analysis
- Loan-purpose analysis
- Portfolio exposure
- Customer risk segmentation
- Fraud-rate analysis
- Fraud exposure
- High-value fraud identification
- Fraud transaction ranking

The SQL layer complements the machine-learning models by providing interpretable portfolio-level analytics and business monitoring capabilities.

In [75]:
from pathlib import Path

project_path = Path(
    r"D:\Banking-Credit-Risk-Fraud-Analytics"
)

print("Root files:")

for item in project_path.iterdir():
    if item.is_file():
        print("📄", item.name)

print("\nDocs files:")

for item in (project_path / "docs").iterdir():
    if item.is_file():
        print("📄", item.name)

Root files:
📄 .gitignore.txt
📄 README.txt
📄 requirements.txt

Docs files:
📄 environment.txt
